<a href="https://colab.research.google.com/github/Shahid-Ahmed-S/RAG/blob/main/Prompt_Engineering_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q groq

from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('gen-ai')
except:
    GROQ_API_KEY = "your-key"

client = Groq(api_key=GROQ_API_KEY)

def ask(prompt):
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model = "openai/gpt-oss-20b"
        #model="llama-3.3-70b-versatile"
    )
    return response.choices[0].message.content

print("✅ Ready!")

✅ Ready!


In [9]:
# Zero-shot: Just ask directly
prompt = "Classify this review as positive or negative: 'The food was terrible and cold.'"
print(ask(prompt))

Negative


In [12]:
# Few-shot: Give examples first
prompt = """
Classify the sentiment:

Review: "Amazing service, loved it!" -> Positive
Review: "Worst experience ever" -> Negative
Review: "It was okay, nothing special" -> Neutral

Review: "The product exceeded my expectations!" ->
"""
print(ask(prompt))

Positive


In [13]:
# Without CoT
print("Without CoT:")
print(ask("What is 23 * 17?"))

# With CoT
print("\nWith CoT:")
cot_prompt = """
What is 23 * 17?

Let's solve this step by step:
"""
print(ask(cot_prompt))

Without CoT:
23 × 17 = 391.

With CoT:
Sure, let’s break it down step by step.

### 1. Write the multiplication in expanded form
\(23 \times 17\) can be split into two parts:
- Multiply 23 by the tens digit of 17 (which is 1, but we treat it as 10 because it’s in the tens place)
- Multiply 23 by the ones digit of 17 (which is 7)

So:
\[
23 \times 17 = 23 \times 10 \;+\; 23 \times 7
\]

### 2. Compute each part

1. **\(23 \times 10\)**  
   \[
   23 \times 10 = 230
   \]

2. **\(23 \times 7\)**  
   \[
   23 \times 7 = 161
   \]
   (You can calculate \(20 \times 7 = 140\) and \(3 \times 7 = 21\), then add them: \(140 + 21 = 161\).)

### 3. Add the results

\[
230 \;+\; 161 \;=\; 391
\]

---

### Alternative: Column multiplication

```
      23
  ×   17
  ------
     161   ← 23 × 7
+   230    ← 23 × 10
  ------
     391
```

Both methods give the same answer.

### Result
\[
23 \times 17 = 391
\]


In [17]:
prompt = """
You are an expert Python programmer with 20 years of experience.
You explain code simply and always provide examples.

Explain why json format is better? Keep the answer in two lines"""
print(ask(prompt))

JSON is lightweight, language‑agnostic, and directly maps to Python dicts/lists (e.g., `{"key":"value"}` ↔ `{"key":"value"}`), simplifying serialization with `json.loads`/`json.dumps`.  
Its minimal syntax (no closing tags) boosts readability and reduces parsing overhead, making it ideal for web APIs.


In [19]:
# Request specific format
prompt = """
Analyze this product review and respond in JSON format:

Review: "Great laptop, fast performance, but battery life could be better. Worth the price."

Respond with this exact JSON structure:
{
  "sentiment": "positive/negative/mixed",
  "pros": ["list of pros"],
  "cons": ["list of cons"],
  "rating": "1-5"
}
"""
print(ask(prompt))

{
  "sentiment": "positive",
  "pros": [
    "Great laptop",
    "Fast performance",
    "Worth the price"
  ],
  "cons": [
    "Battery life could be better"
  ],
  "rating": "4"
}


In [20]:
# Create a reusable prompt library
class PromptLibrary:

    @staticmethod
    def summarize(text, length="short"):
        return f"Summarize this text in a {length} paragraph:\n\n{text}"

    @staticmethod
    def translate(text, target_lang):
        return f"Translate to {target_lang}:\n\n{text}"

    @staticmethod
    def code_explain(code):
        return f"Explain this code line by line:\n\n```\n{code}\n```"

    @staticmethod
    def sentiment(text):
        return f"""Analyze sentiment. Respond with JSON:
{{"sentiment": "positive/negative/neutral", "confidence": "high/medium/low", "reason": "brief reason"}}

Text: {text}"""

    @staticmethod
    def quiz_question(topic, difficulty="medium"):
        return f"""Create a {difficulty} difficulty quiz question about {topic}.
Format:
Question: [question]
A) [option]
B) [option]
C) [option]
D) [option]
Answer: [letter]"""

# Test the library
pl = PromptLibrary()
print(ask(pl.quiz_question("Python loops", "easy")))

Question: What keyword starts a for loop in Python?  
A) repeat  
B) for  
C) loop  
D) iterate  
Answer: B


In [ ]:
class PromptTemplate:
    def __init__(self, template):
        self.template = template

    def format(self, **kwargs):
        return self.template.format(**kwargs)

    def run(self, **kwargs):
        prompt = self.format(**kwargs)
        return ask(prompt)

# Create templates
email_template = PromptTemplate("""
Write a {tone} email about {topic}.
- Keep it {length}
- Include a clear call to action
""")

# Use the template
result = email_template.run(
    tone="professional",
    topic="project deadline extension",
    length="brief (3-4 sentences)"
)
print(result)